### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [9]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is missing. Check your .env file.")

model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0
)

response = model.invoke("Why do parrots talk?")
print(response.content)

Parrots don’t “talk” in the way humans use language, but many species are excellent mimics and can reproduce human speech sounds. Their ability to do this comes from a combination of anatomy, brain wiring, and social behavior:

| Factor | How it helps parrots mimic speech |
|--------|-----------------------------------|
| **Vocal anatomy** | Parrots have a highly flexible syrinx (the bird equivalent of a voice box) and a muscular tongue and beak that let them shape a wide range of sounds, including the consonants and vowels we use in speech. |
| **Auditory learning** | Their ears are tuned to pick up subtle variations in pitch and rhythm, so they can copy the exact cadence of a word or phrase they hear repeatedly. |
| **Social nature** | In the wild, parrots live in flocks where they constantly exchange calls to keep the group together. Mimicking sounds is a way to bond, attract attention, and signal identity. When kept as pets, they treat their human caretakers as part of their “flock

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [11]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_d52fc9f6-0f09-4edd-a667-582e0196c895', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 127, 'total_tokens': 166, 'completion_time': 0.082443943, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.006019494, 'prompt_tokens_details': None, 'queue_time': 0.210980006, 'total_time': 0.088463437}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c3da0d4bb9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0f320-efc8-7cf1-b7ba-d637e0089955-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_d52fc9f6-0f09-4edd-a667-582e0196c895', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 

### Tool Execution Loops

In [12]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny.


In [13]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_3f6a972d-befe-498a-b4ac-0c7d138ebc8b', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 126, 'total_tokens': 166, 'completion_time': 0.085979093, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006980053, 'prompt_tokens_details': None, 'queue_time': 0.348971421, 'total_time': 0.092959146}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f320-f199-7e01-a03f-e713524a13f6-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_3f6a972d-befe-498a-b4ac-0c7d138eb